> **Note — notebook surface is moving.** Starting with `v0.4.0`, all notebooks
> in this repository will move to the dedicated
> [`forecastability-examples`](https://github.com/example/forecastability-examples)
> sibling repository. The library itself will keep only deterministic Python
> APIs, scripts under `scripts/`, and recipe pages under `docs/recipes/`.
> See [docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md](../../docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md).

<!-- type: tutorial -->
# Lagged Exogenous Triage Showcase (v0.3.2)

This tutorial keeps a deterministic, triage-first workflow:
1. Diagnose lag-domain structure and role semantics.
2. Build a sparse lag map for downstream forecasting hand-off.
3. Keep model fitting out of scope for this package.

Required sections covered in order:
1. Multivariate vs covariant-informative role taxonomy
2. Standard cross-correlation baseline on panel
3. Extended cross_ami profile including lag=0 diagnostic
4. Sparse lag selection with xami_sparse
5. Known-future opt-in worked example
6. DTW omission rationale
7. Hand-off to downstream tensor builder

## Setup

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from forecastability import (
    generate_known_future_calendar_pair,
    generate_lagged_exog_panel,
    run_lagged_exogenous_triage,
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)

OUTPUT_ROOT = Path("outputs/notebooks/walkthroughs/03_lagged_exogenous_triage_showcase")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
N = 1500
MAX_LAG = 6
N_SURROGATES = 99
KNOWN_FUTURE_DRIVER = "known_future_calendar"

DRIVER_ORDER = [
    "direct_lag2",
    "mediated_lag1",
    "redundant",
    "noise",
    "instant_only",
    "nonlinear_lag1",
    KNOWN_FUTURE_DRIVER,
]

## 1. Multivariate vs covariant-informative role taxonomy

Multivariate forecasting tensors and covariant-informative triage have different jobs:

- **Covariant-informative triage** (this package): diagnose whether each driver-lag pair is informative, mediated, redundant, diagnostic-only, or known-future.
- **Multivariate model training** (downstream framework): consume only selected predictive/known-future lag features and fit the forecast model.

For lagged exogenous triage, the role labels are explicit:
- `lag_role='instant'` at `lag=0` and `lag_role='predictive'` for `lag>=1`.
- `tensor_role` marks `diagnostic`, `predictive`, or `known_future` eligibility for hand-off.

In [ ]:
panel = generate_lagged_exog_panel(n=N, seed=SEED)
target = panel["target"].to_numpy()
drivers = {name: panel[name].to_numpy() for name in DRIVER_ORDER}

bundle = run_lagged_exogenous_triage(
    target,
    drivers,
    target_name="target",
    max_lag=MAX_LAG,
    n_surrogates=N_SURROGATES,
    random_state=SEED,
    known_future_drivers={KNOWN_FUTURE_DRIVER: True},
)

lag0_rows = [row for row in bundle.profile_rows if row.lag == 0]
taxonomy_frame = pd.DataFrame(
    [
        {
            "driver": row.driver,
            "lag": row.lag,
            "lag_role": row.lag_role,
            "tensor_role": row.tensor_role,
        }
        for row in sorted(lag0_rows, key=lambda item: item.driver)
    ]
)

display(Markdown("**Lag-0 taxonomy rows (diagnostic vs known_future):**"))
display(taxonomy_frame)

selected_total = sum(1 for row in bundle.selected_lags if row.selected_for_tensor)
print(f"Selected lag rows for downstream hand-off: {selected_total}")
print(f"Known-future opt-in drivers: {bundle.known_future_drivers}")

## 2. Standard cross-correlation baseline on panel

The cross-correlation profile is a linear baseline. It is useful as a fast screening view,
but it is not causal evidence and can miss nonlinear structure.

In [ ]:
corr_rows = [
    row
    for row in bundle.profile_rows
    if row.correlation is not None
]
corr_frame = pd.DataFrame(
    [
        {
            "driver": row.driver,
            "lag": row.lag,
            "correlation": float(row.correlation),
            "abs_correlation": abs(float(row.correlation)),
        }
        for row in corr_rows
    ]
)

corr_peak = (
    corr_frame.sort_values(["driver", "abs_correlation", "lag"], ascending=[True, False, True])
    .groupby("driver", as_index=False)
    .first()
    .sort_values("driver")
)
display(Markdown("**Best |correlation| lag per driver (baseline):**"))
display(corr_peak[["driver", "lag", "correlation", "abs_correlation"]])

plot_drivers = ["direct_lag2", "mediated_lag1", "instant_only"]
fig, ax = plt.subplots(figsize=(10, 4))
for driver in plot_drivers:
    sub = corr_frame[corr_frame["driver"] == driver].sort_values("lag")
    ax.plot(sub["lag"], sub["correlation"], marker="o", label=driver)
ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1.0, alpha=0.7)
ax.set_title("Cross-correlation baseline (selected drivers)")
ax.set_xlabel("Lag")
ax.set_ylabel("Correlation")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Extended cross_ami profile including lag=0 diagnostic

The lag-domain profile includes `lag=0` by default. That row is kept as an instant-impact diagnostic and separated from predictive lags in interpretation.

In [ ]:
ami_rows = [row for row in bundle.profile_rows if row.cross_ami is not None]
ami_frame = pd.DataFrame(
    [
        {
            "driver": row.driver,
            "lag": row.lag,
            "cross_ami": float(row.cross_ami),
            "significance": row.significance,
            "significance_source": row.significance_source,
            "lag_role": row.lag_role,
            "tensor_role": row.tensor_role,
        }
        for row in ami_rows
    ]
)

lag0_diagnostic = ami_frame[ami_frame["lag"] == 0].sort_values("driver")
display(Markdown("**lag=0 cross_ami diagnostics:**"))
display(lag0_diagnostic[["driver", "cross_ami", "lag_role", "tensor_role", "significance"]])

ami_peak = (
    ami_frame.sort_values(["driver", "cross_ami", "lag"], ascending=[True, False, True])
    .groupby("driver", as_index=False)
    .first()
    .sort_values("driver")
)
display(Markdown("**Best cross_ami lag per driver (extended with lag=0):**"))
display(ami_peak[["driver", "lag", "cross_ami", "significance"]])

fig, ax = plt.subplots(figsize=(10, 4))
for driver in ["direct_lag2", "mediated_lag1", "nonlinear_lag1"]:
    sub = ami_frame[ami_frame["driver"] == driver].sort_values("lag")
    ax.plot(sub["lag"], sub["cross_ami"], marker="o", label=driver)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1.0, alpha=0.7)
ax.set_title("Extended cross_ami profile (lag=0 diagnostic included)")
ax.set_xlabel("Lag")
ax.set_ylabel("cross_ami")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Sparse lag selection with xami_sparse

`xami_sparse` performs deterministic greedy lag pruning over predictive lags (`lag>=1`).
The output rows already include selector metadata and tensor-role intent for hand-off.

In [ ]:
selected_rows = [row for row in bundle.selected_lags if row.selected_for_tensor]
selection_frame = pd.DataFrame(
    [
        {
            "driver": row.driver,
            "lag": row.lag,
            "selector_name": row.selector_name,
            "score": row.score,
            "selection_order": row.selection_order,
            "tensor_role": row.tensor_role,
        }
        for row in selected_rows
    ]
)
selection_frame = selection_frame.sort_values(["driver", "lag"]).reset_index(drop=True)

display(Markdown("**Sparse selected lag rows (selected_for_tensor=True):**"))
display(selection_frame)
print("selector_name values:", sorted(selection_frame["selector_name"].unique()))

## 5. Known-future opt-in worked example

Lag-0 selection is blocked by default. It is only allowed when a driver is explicitly declared known future at prediction time.

Cross-links:
- Python example: [examples/covariant_informative/lagged_exogenous/lagged_exog_python_example.py](../../examples/covariant_informative/lagged_exogenous/lagged_exog_python_example.py)
- Known-future example: [examples/covariant_informative/lagged_exogenous/known_future_opt_in_example.py](../../examples/covariant_informative/lagged_exogenous/known_future_opt_in_example.py)
- Showcase script: [scripts/run_showcase_lagged_exogenous.py](../../scripts/run_showcase_lagged_exogenous.py)

In [ ]:
known_panel = generate_known_future_calendar_pair(n=900, seed=SEED)
known_target = known_panel["target"].to_numpy()
known_driver = {KNOWN_FUTURE_DRIVER: known_panel[KNOWN_FUTURE_DRIVER].to_numpy()}

default_bundle = run_lagged_exogenous_triage(
    known_target,
    known_driver,
    target_name="target",
    max_lag=4,
    n_surrogates=N_SURROGATES,
    random_state=SEED,
)
opt_in_bundle = run_lagged_exogenous_triage(
    known_target,
    known_driver,
    target_name="target",
    max_lag=4,
    n_surrogates=N_SURROGATES,
    random_state=SEED,
    known_future_drivers={KNOWN_FUTURE_DRIVER: True},
)

default_lag0 = sum(
    1
    for row in default_bundle.selected_lags
    if row.selected_for_tensor and row.driver == KNOWN_FUTURE_DRIVER and row.lag == 0
)
opt_in_lag0 = sum(
    1
    for row in opt_in_bundle.selected_lags
    if row.selected_for_tensor and row.driver == KNOWN_FUTURE_DRIVER and row.lag == 0
)

comparison = pd.DataFrame(
    [
        {
            "run": "default",
            "known_future_drivers": default_bundle.known_future_drivers,
            "selected_lag0_count": default_lag0,
        },
        {
            "run": "opt_in",
            "known_future_drivers": opt_in_bundle.known_future_drivers,
            "selected_lag0_count": opt_in_lag0,
        },
    ]
)
display(comparison)
display(Markdown("**Result:** lag=0 selection appears only under explicit known-future opt-in."))

## 6. DTW omission rationale (text only)

Dynamic Time Warping (DTW) is intentionally omitted from this triage surface.

Rationale:
- Lagged-exogenous triage here is fixed-lag and deterministic; DTW introduces elastic alignment that changes temporal semantics.
- The selection target is a sparse, interpretable lag map for downstream feature construction, not sequence-alignment similarity.
- DTW scores are not directly comparable to the current cross-correlation and cross_ami significance framing used in this release.

So for v0.3.2, DTW is out-of-scope by design for lag-domain triage and hand-off contracts.

## 7. Hand-off: sparse lag map consumed by downstream tensor builder

The triage artifact is the selected lag map. A downstream model framework consumes this map to construct lagged feature tensors.

In [ ]:
selected_for_tensor = [row for row in bundle.selected_lags if row.selected_for_tensor]
selected_lag_map: dict[str, list[int]] = {}
for row in sorted(selected_for_tensor, key=lambda item: (item.driver, item.lag)):
    selected_lag_map.setdefault(row.driver, []).append(int(row.lag))

handoff_payload = {
    "target_name": bundle.target_name,
    "max_lag": bundle.max_lag,
    "known_future_drivers": bundle.known_future_drivers,
    "selected_lag_map": selected_lag_map,
}

display(Markdown("**Downstream hand-off payload (deterministic triage output):**"))
display(pd.Series(handoff_payload, name="value"))

handoff_path = OUTPUT_ROOT / "selected_lag_map_handoff.json"
pd.Series(handoff_payload).to_json(handoff_path, indent=2)
print(f"Wrote hand-off payload: {handoff_path}")